<a href="https://colab.research.google.com/github/Manawariqbal/LLM-FIneTuning/blob/main/Knowledge_Distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#install libraries
!pip install -q transformers datasets accelerate

In [3]:
#Import
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score

In [5]:
#Load a Small Dataset
# Load SST-2 and take a small subset for the demo
dataset = load_dataset("nyu-mll/glue", "sst2")
train_dataset = dataset["train"].select(range(2000))
eval_dataset = dataset["validation"].select(range(500))

print(train_dataset)
print(train_dataset[0])

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 2000
})
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [7]:
#Tokenizer + Teacher Model
teacher_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

def tokenize(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset = eval_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load teacher (we will fine-tune it a little first so it becomes a good teacher)
teacher = AutoModelForSequenceClassification.from_pretrained(teacher_name, num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
#Quick Fine-tune the Teacher (Optional but Recommended)
training_args = TrainingArguments(
    output_dir="./teacher",
    per_device_train_batch_size=16,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=teacher,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()
teacher.save_pretrained("./teacher_finetuned")
print("Teacher fine-tuned and saved.")

Step,Training Loss


KeyboardInterrupt: 

In [9]:
#Load Student Model
student_name = "distilbert-base-uncased"
student = AutoModelForSequenceClassification.from_pretrained(student_name, num_labels=2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
#Custom Distillation Trainer
class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, temperature=2.0, alpha=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.alpha = alpha
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")

        # Student forward
        outputs = model(**inputs)
        student_logits = outputs.logits

        # Teacher forward (no gradient)
        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits

        # Hard label loss
        loss_ce = F.cross_entropy(student_logits, labels)

        # Soft label (distillation) loss
        student_soft = F.log_softmax(student_logits / self.temperature, dim=-1)
        teacher_soft = F.softmax(teacher_logits / self.temperature, dim=-1)
        loss_kd = F.kl_div(student_soft, teacher_soft, reduction="batchmean") * (self.temperature ** 2)

        # Combined loss
        loss = self.alpha * loss_ce + (1 - self.alpha) * loss_kd

        return (loss, outputs) if return_outputs else loss

In [ ]:
# Redefining DistillationTrainer with compatibility for newer transformers versions
class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, temperature=2.0, alpha=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.alpha = alpha
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        # Student forward
        outputs = model(**inputs)
        student_logits = outputs.logits

        # Teacher forward (no gradient)
        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits

        # Hard label loss
        loss_ce = F.cross_entropy(student_logits, labels)

        # Soft label (distillation) loss
        student_soft = F.log_softmax(student_logits / self.temperature, dim=-1)
        teacher_soft = F.softmax(teacher_logits / self.temperature, dim=-1)
        loss_kd = F.kl_div(student_soft, teacher_soft, reduction="batchmean") * (self.temperature ** 2)

        # Combined loss
        loss = self.alpha * loss_ce + (1 - self.alpha) * loss_kd

        return (loss, outputs) if return_outputs else loss

#Train the Student with Distillation
# Reload the fine-tuned teacher
#teacher = AutoModelForSequenceClassification.from_pretrained("./teacher_finetuned")
teacher = AutoModelForSequenceClassification.from_pretrained(teacher_name, num_labels=2)
training_args = TrainingArguments(
    output_dir="./student_distilled",
    per_device_train_batch_size=16,
    num_train_epochs=2,
    learning_rate=3e-5,
    logging_steps=50,
    eval_strategy="epoch",
    report_to="none"
)

distill_trainer = DistillationTrainer(
    model=student,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    teacher_model=teacher,
    temperature=2.0,   # try 1.0, 2.0, 4.0
    alpha=0.5          # balance between hard and soft loss
)

distill_trainer.train()
student.save_pretrained("./student_distilled")
print("Distilled student saved.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3

Epoch,Training Loss,Validation Loss


In [ ]:
#Compare Results (Optional but Useful)
# Baseline student (normal training, no distillation)
baseline = AutoModelForSequenceClassification.from_pretrained(student_name, num_labels=2)

baseline_trainer = Trainer(
    model=baseline,
    args=TrainingArguments(
        output_dir="./baseline",
        per_device_train_batch_size=16,
        num_train_epochs=2,
        learning_rate=3e-5,
        report_to="none"
    ),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

baseline_trainer.train()

def compute_accuracy(model, dataset):
    model.eval()
    preds, labels = [], []
    loader = DataLoader(dataset, batch_size=32)
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(model.device) for k, v in batch.items()}
            logits = model(**{k: batch[k] for k in ["input_ids", "attention_mask"]}).logits
            preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            labels.extend(batch["label"].cpu().numpy())
    return accuracy_score(labels, preds)

print("Teacher accuracy:", compute_accuracy(teacher, eval_dataset))
print("Distilled student accuracy:", compute_accuracy(student, eval_dataset))
print("Baseline student accuracy:", compute_accuracy(baseline, eval_dataset))